# Explaining TSlib models with WinTSR[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/tslib_models.ipynb)The [quickstart](quickstart.ipynb) explains a plain model that takes`(batch, seq_len, n_features)`. Real forecasting models from[Time-Series-Library (TSlib)](https://github.com/thuml/Time-Series-Library) --DLinear, iTransformer, TimesNet, Autoformer and the rest -- take **four** tensors:```pythonmodel(x_enc, x_mark_enc, x_dec, x_mark_dec)```This notebook shows how to point WinTSR at them. **No wrapper class is needed**: yousplit the four arguments into the ones you want attributed and the ones that are justcontext.

In [ ]:
# tslens (with WinTSR), plus the model zoo from the WinTSR-research repo (the paper's# training/interpretation harness) purely as a source of TSlib model definitions%pip install -q tslens einops reformer-pytorch!git clone -q https://github.com/khairulislam/WinTSR-research.git%cd WinTSR-research

## 1. Build a TSlib modelTSlib models are configured with a single `configs` object. We use a small one here sothe notebook runs on CPU; a trained checkpoint works exactly the same way.

In [ ]:
import typesimport torchfrom models import DLinearBATCH, SEQ_LEN, LABEL_LEN, PRED_LEN, N_FEAT = 4, 24, 6, 6, 3configs = types.SimpleNamespace(    task_name="long_term_forecast",    seq_len=SEQ_LEN, label_len=LABEL_LEN, pred_len=PRED_LEN,    enc_in=N_FEAT, dec_in=N_FEAT, c_out=N_FEAT,    moving_avg=25, features="S", output_attention=False,    d_model=16, n_heads=2, e_layers=1, d_layers=1, d_ff=32,    dropout=0.1, activation="gelu", factor=1, embed="timeF", freq="h",    num_class=2, use_norm=True, class_strategy="projection",)model = DLinear.Model(configs).eval()# The four tensors a TSlib forward() expectsx_enc      = torch.randn(BATCH, SEQ_LEN, N_FEAT)              # the series itselfx_mark_enc = torch.randn(BATCH, SEQ_LEN, 4)                   # calendar featuresx_dec      = torch.zeros(BATCH, LABEL_LEN + PRED_LEN, N_FEAT) # decoder inputx_mark_dec = torch.randn(BATCH, LABEL_LEN + PRED_LEN, 4)      # decoder calendarwith torch.no_grad():    out = model(x_enc, x_mark_enc, x_dec, x_mark_dec)print("forward output:", tuple(out.shape), "= (batch, pred_len, c_out)")

## 2. Split the argumentsThis is the only thing you need to get right:| TSlib argument | Where it goes | Why || --- | --- | --- || `x_enc` | `inputs` | the series you want explained || `x_mark_enc` | `inputs` | calendar features, also attributed || `x_dec` | `additional_forward_args` | context, held fixed || `x_mark_dec` | `additional_forward_args` | context, held fixed |Anything in `inputs` gets perturbed and receives attributions. Anything in`additional_forward_args` is passed through to the model untouched.

In [ ]:
from tslens import WinTSRinputs    = (x_enc, x_mark_enc)baselines = (torch.zeros_like(x_enc), torch.zeros_like(x_mark_enc))context   = (x_dec, x_mark_dec)attr_enc, attr_mark = WinTSR(model).attribute(    inputs=inputs,    baselines=baselines,    additional_forward_args=context,    threshold=0.5,)print("attr for x_enc     :", tuple(attr_enc.shape))print("attr for x_mark_enc:", tuple(attr_mark.shape))

Because `inputs` was a tuple, you get a tuple of attributions back, aligned with it.`attr_enc` is usually the one you care about.**Note the output dimension.** The model returns `(batch, pred_len, c_out)`, whichattribution flattens into `n_output = pred_len * c_out` separate outputs. With`pred_len=6` and `c_out=3` that is 18, so the attribution shape is`(batch, 18, seq_len, n_features)` -- one saliency map per predicted value.

In [ ]:
n_output = PRED_LEN * configs.c_outassert attr_enc.shape == (BATCH, n_output, SEQ_LEN, N_FEAT)print(f"{PRED_LEN} horizons x {configs.c_out} channels = {n_output} outputs, "      f"each with a {SEQ_LEN} x {N_FEAT} saliency map")

## 3. Plot itAverage over the output dimension for an overall view, or index it to explain a single horizon.

In [ ]:
import matplotlib.pyplot as pltsaliency = attr_enc.abs().mean(dim=1).detach()   # average over the 18 outputsfig, axes = plt.subplots(1, 3, figsize=(13, 3))for ax, i in zip(axes, range(3)):    im = ax.imshow(saliency[i].T, aspect="auto", cmap="viridis")    ax.set_title(f"sample {i}")    ax.set_xlabel("time step")    ax.set_ylabel("feature")    ax.set_yticks(range(N_FEAT))plt.tight_layout()plt.show()

## 4. It is the same for every other TSlib modelThe calling convention does not change -- only the model constructor does.

In [ ]:
from models import iTransformerfor name, module in [("DLinear", DLinear), ("iTransformer", iTransformer)]:    m = module.Model(configs).eval()    a, _ = WinTSR(m).attribute(        inputs=inputs, baselines=baselines,        additional_forward_args=context, threshold=0.5,    )    print(f"{name:<14} -> {tuple(a.shape)}")

## Models that take a single input`CALF` and `OFA` (GPT4TS) consume only `x_enc`. For those, drop the tuple entirely:```pythonattr = WinTSR(model).attribute(    inputs=x_enc,    baselines=torch.zeros_like(x_enc),    threshold=0.5,)```## Your own modelIf your model already takes `(batch, seq_len, n_features)` and returns predictions,there is nothing to configure -- see the [quickstart](quickstart.ipynb):```pythonattr = WinTSR(model).attribute(inputs, baselines=torch.zeros_like(inputs))```